# Project Additional Materials — Baseline AutoML (Monthly)

- Student ID: 10841269  
- Course Code: DATA70132  
- Academic Year: 2024–25  

**Environment:** See `README` and `ERP_Environment_2025.yaml`.  
**Reproduction:** Run this notebook top to bottom in the same folder as `All_UKonly_monthly_cleaned.nc`.  

This notebook trains a FLAML baseline on monthly data and saves the fitted model and log.


## Step 0 — Import required libraries
Load the Python packages required for the monthly FLAML baseline.  
(Dependencies are listed in the `README` and `ERP_Environment_2025.yaml`.)


In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import random
import warnings
import pickle
from flaml import AutoML
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings('ignore')


## Step 1 — Train monthly baseline with FLAML
1. Set seeds.  
2. Load the preprocessed monthly UK-only dataset (`All_UKonly_monthly_cleaned.nc`) and drop rows with missing values.  
3. Define features and target as in the report; use a time-based training window: `year_month` 200601–202012.  
4. Train with k-fold CV and log to file.  
5. Print the best model summary and save the fitted model (pickle).


In [ ]:
# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

# Load dataset
ds = xr.open_dataset("All_UKonly_monthly_cleaned.nc", decode_times=False)
df = ds.to_dataframe().dropna().reset_index()

# Time split
df_train = df[(df["year_month"] >= 200601) & (df["year_month"] <= 202012)]

features = ["TREFHT", "FLNS", "FSNS", "QBOT", "UBOT", "VBOT", "PRECT", "PRSN"]
target = "TREFMXAV_U"

X_train, y_train = df_train[features], df_train[target]

# FLAML baseline
automl = AutoML()
automl.fit(
    X_train, y_train,
    task="regression",
    time_budget=3600,          
    metric="r2",         
    eval_method="cv",
    n_jobs=-1,  
    log_file_name="automl_training_monthly.log",
)

# Best model info
print("Best model by FLAML:", automl.best_estimator)
print("Best config:", automl.best_config)
print("Best validation loss:", automl.best_loss)

model_save_path = 'automl_training_monthly.pkl'
with open(f"{model_save_path}", "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"Model for UK urban temperature prediction saved as {model_save_path}")


[flaml.automl.logger: 08-16 14:07:30] {1752} INFO - task = regression
[flaml.automl.logger: 08-16 14:07:30] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 08-16 14:07:30] {1862} INFO - Minimizing error metric: 1-r2
[flaml.automl.logger: 08-16 14:07:30] {1979} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'catboost']
[flaml.automl.logger: 08-16 14:07:30] {2282} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 08-16 14:07:32] {2417} INFO - Estimated sufficient time budget=12597s. Estimated necessary time budget=109s.
[flaml.automl.logger: 08-16 14:07:32] {2466} INFO -  at 1.7s,	estimator lgbm's best error=0.4977,	best estimator lgbm's best error=0.4977
[flaml.automl.logger: 08-16 14:07:32] {2282} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 08-16 14:07:33] {2466} INFO -  at 2.9s,	estimator lgbm's best error=0.4977,	best estimator lgbm's best error=0.4977
[flaml.automl.logger: 08-16 14: